# VisionFromScratch — Colab 啟動器

這個 notebook **只負責準備環境與啟動訓練**，所有訓練邏輯都在 GitHub repo 的 `src/` 內。

流程：檢查 GPU → 掛載 Drive → 取得程式碼 → 安裝套件 → 解壓資料集 → 開 TensorBoard → 訓練 → 評估

斷線後重新執行全部 cell 即可，訓練會自動從 Drive 上最新的 checkpoint 接著跑。

In [ ]:
# 1. 確認 GPU（建議：執行階段 → 變更執行階段類型 → T4 GPU）
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. 設定
REPO_URL = "https://github.com/YiLongSun/VisionFromScratch"
BRANCH = "main"
CONFIG = "configs/detection.yaml"        # configs/detection.yaml 或 configs/segmentation.yaml
EXTRA_ARGS = ""                              # 額外覆寫，例如 "optim.lr=0.005 data.batch_size=4"

DRIVE_ZIP = "/content/drive/MyDrive/ColabNotebooks/Datasets/VOC0712.zip"         # Drive 上的資料集 zip
OUTPUT_ROOT = "/content/drive/MyDrive/ColabNotebooks/VisionFromScratch/outputs"  # checkpoint 與 TensorBoard
DATA_DIR = "/content/data"                                        # 解壓到本機（讀取比 Drive 快很多）
REPO_DIR = "/content/VisionFromScratch"

In [ ]:
# 3. 掛載 Google Drive
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# 4. 取得程式碼（已存在則更新到最新版）
import os
if os.path.isdir(f"{REPO_DIR}/.git"):
    !git -C {REPO_DIR} fetch -q origin {BRANCH} && git -C {REPO_DIR} checkout -q {BRANCH} && git -C {REPO_DIR} reset -q --hard origin/{BRANCH}
else:
    !git clone -q -b {BRANCH} {REPO_URL} {REPO_DIR}
!git -C {REPO_DIR} log -1 --oneline

In [ ]:
# 5. 安裝套件（torch / torchvision 使用 Colab 內建版本）
!pip install -q -r {REPO_DIR}/requirements.txt
import torch, torchvision, timm
print(torch.__version__, torchvision.__version__, timm.__version__, "CUDA:", torch.cuda.is_available())

In [ ]:
# 6. 準備資料集：先複製 zip 到本機再解壓（直接從 Drive 解壓會非常慢）
import os, shutil, time
marker = f"{DATA_DIR}/.voc_ready"
if os.path.exists(marker):
    print("資料集已就緒，略過")
else:
    os.makedirs(DATA_DIR, exist_ok=True)
    start = time.time()
    shutil.copy(DRIVE_ZIP, "/content/VOC0712.zip")
    print(f"複製完成（{time.time() - start:.0f} 秒），開始解壓…")
    !unzip -q -o /content/VOC0712.zip -d {DATA_DIR}
    os.remove("/content/VOC0712.zip")
    open(marker, "w").close()
    print(f"完成，共 {time.time() - start:.0f} 秒")
!ls {DATA_DIR}/VOC0712/VOC_dataset/VOCdevkit

In [ ]:
# 7. TensorBoard（log 在 Drive 上，所有實驗都看得到）
%reload_ext tensorboard
%tensorboard --logdir {OUTPUT_ROOT}

In [ ]:
# 8. 訓練（若 Drive 上已有 checkpoint 會自動續訓）
%cd {REPO_DIR}
!python -m src.train --config {CONFIG} experiment.output_root={OUTPUT_ROOT} data.root={DATA_DIR}/VOC0712/VOC_dataset/VOCdevkit {EXTRA_ARGS}

In [ ]:
# 9. 用 best.pt 在完整驗證集上評估
%cd {REPO_DIR}
!python -m src.evaluate --config {CONFIG} --checkpoint best experiment.output_root={OUTPUT_ROOT} data.root={DATA_DIR}/VOC0712/VOC_dataset/VOCdevkit